# The Model Context Protocol (MCP)

> An open standard for connecting LLM applications to external tools, data, and context.
> Spec & docs: https://modelcontextprotocol.io

## Why MCP exists
Before MCP, every LLM app integrated with every external system through bespoke glue code — an **M×N problem** (M apps × N data sources/tools). MCP collapses that into an **M+N problem**: each app speaks MCP once, each tool/data source implements MCP once, and they all interoperate. Think of it as a **"USB-C port for AI"**.

## What this notebook covers
A complete, **runnable** tour of the protocol — no API keys, no model downloads. Every concept is demonstrated live using an in-memory client↔server session with *mocked* client capabilities:

- **Server primitives:** Tools, Resources, Prompts
- **Client primitives:** Sampling, Roots, Elicitation
- **Mechanics:** the `initialize` / capability handshake, transports, logging & notifications, and the JSON-RPC message format
- **Authentication:** a simple bearer-token verifier for HTTP-served servers

> This is a guided walkthrough — the **starter and solution notebooks are identical**. Read top to bottom and run each cell.

## Learning Objectives
By the end you will be able to:
- Explain MCP's **host / client / server** architecture and the role of JSON-RPC 2.0
- Build an MCP server with `FastMCP` exposing **tools, resources, and prompts**
- Drive a server from a client: **initialize, list, and invoke** each primitive
- Describe the client-side primitives — **sampling, roots, elicitation** — and see them run
- Recognize the **transports** and **message format** used to deploy a real server

## Setup
MCP ships an official Python SDK (the `mcp` package), which requires Python 3.10+. It includes both the server framework (`FastMCP`) and the client (`ClientSession`).

In [1]:
!pip install -q "mcp>=1.2,<2"  # notebook uses the v1 API (FastMCP); mcp 2.x renamed it

In [2]:
import logging
from pydantic import BaseModel
import mcp.types as types
from mcp.server.fastmcp import FastMCP, Context
from mcp.shared.memory import create_connected_server_and_client_session as connect_client

# The MCP server logs every request at INFO level; quiet it for a cleaner notebook.
logging.getLogger("mcp").setLevel(logging.WARNING)

print("✓ MCP SDK imported")

✓ MCP SDK imported


## Part 1 — Architecture

MCP is a **client–server** protocol with three roles:

```
┌──────────────────────────────────────────────┐
│ HOST   (e.g. Claude Desktop, an IDE)           │
│                                                │
│   ┌──────────┐         ┌──────────┐            │
│   │ Client A │◄───MCP──►│ Server 1 │  filesystem│
│   └──────────┘         └──────────┘            │
│   ┌──────────┐         ┌──────────┐            │
│   │ Client B │◄───MCP──►│ Server 2 │  GitHub    │
│   └──────────┘         └──────────┘            │
│        ▲                                       │
│        │ embeds the LLM                        │
└────────┼───────────────────────────────────────┘
         │
       (user)
```

- **Host** — the LLM application the user interacts with. It embeds the model and runs one or more clients.
- **Client** — lives inside the host and holds a **1:1 connection** to a single server.
- **Server** — a program exposing capabilities (tools / resources / prompts) over MCP.

**Message format.** All communication is **JSON-RPC 2.0**: *requests* (expect a response), *responses*, and *notifications* (fire-and-forget, no reply).

**Transports.** The same messages travel over either:
- **stdio** — the server runs as a local subprocess; messages flow over stdin/stdout. Best for local tools.
- **Streamable HTTP** — the server runs remotely; messages flow over HTTP with streaming. Best for hosted services.

**Lifecycle.** Every connection opens with an **`initialize`** handshake where both sides advertise what they support (**capability negotiation**), then they exchange messages, then shut down.

## Part 2 — The Primitive Map

MCP is **symmetric**: servers expose primitives to clients, and clients expose primitives back to servers. The clarifying question for each is *who controls it*.

### Server primitives (a server offers these)
| Primitive | Controlled by | REST analogy | Purpose |
|---|---|---|---|
| **Tools** | the **model** | `POST` | Functions the LLM may call to act or compute |
| **Resources** | the **application** | `GET` | Read-only context, addressed by a URI |
| **Prompts** | the **user** | — | Reusable templates / workflows the user triggers |

### Client primitives (a client offers these back)
| Primitive | Initiated by | Purpose |
|---|---|---|
| **Sampling** | server → client's LLM | Server asks the host's model to generate a completion |
| **Roots** | client → server | Client tells the server which filesystem/URI boundaries to work within |
| **Elicitation** | server → user | Server asks the user for additional input mid-task |

### Mechanics (cross-cutting)
Lifecycle & capability negotiation · logging · notifications (list-changed, progress, cancellation) · completion (autocomplete) · pagination · resource subscriptions · ping.

We'll build and run an example of each below.

## Part 3 — Build a Server with `FastMCP`

`FastMCP` is the high-level SDK API: you decorate plain Python functions and it generates the JSON schemas, registration, and dispatch for you. We'll grow **one** server object across the next few cells.

In [3]:
server = FastMCP("demo-server")

# ── TOOLS ──  model-controlled actions; the type hints become the input schema.
@server.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@server.tool()
def verify_arithmetic(expression: str) -> bool:
    """Check whether a simple 'a op b = c' equation is correct (e.g. '6 * 4 = 24')."""
    import re
    m = re.match(r"\s*([\d.]+)\s*([+\-*/])\s*([\d.]+)\s*=\s*([\d.]+)\s*$", expression)
    if not m:
        return False
    a, op, b, c = float(m[1]), m[2], float(m[3]), float(m[4])
    value = {"+": a + b, "-": a - b, "*": a * b, "/": (a / b if b else None)}[op]
    return value is not None and abs(value - c) < 1e-9

print("✓ Registered tools: add, verify_arithmetic")

✓ Registered tools: add, verify_arithmetic


### Tools — *model-controlled*
A tool is a function the **model** decides to invoke. The SDK turns the function signature and docstring into a JSON schema the model sees, so it knows the tool's name, arguments, and purpose. Tools are like `POST` endpoints: they *do* something (compute, call an API, write data). A well-designed host keeps a human in the loop to approve tool calls.

In [4]:
# ── RESOURCES ──  application-controlled, read-only context addressed by URI.

@server.resource("config://server")
def server_config() -> str:
    """A STATIC resource — always the same URI, returns fixed text."""
    return "name=demo-server\nversion=1.0\nmax_results=10"

@server.resource("note://{topic}")
def note(topic: str) -> str:
    """A resource TEMPLATE — the {topic} segment is filled in per request."""
    return f"This is the note about '{topic}'."

print("✓ Registered resources: config://server  and  note://{topic}")

✓ Registered resources: config://server  and  note://{topic}


### Resources — *application-controlled*
A resource exposes **read-only data** identified by a **URI** (`config://server`, `note://daily`, `file:///path`). They behave like `GET` endpoints: fetching one should have no side effects. The **application** decides when to pull a resource into the model's context (whereas *tools* are triggered by the model). A **resource template** like `note://{topic}` is a parameterized URI — one definition that serves many addresses.

In [5]:
# ── PROMPTS ──  user-controlled reusable templates.

@server.prompt()
def review_code(code: str) -> str:
    """A prompt template the user can invoke (e.g. as a slash-command)."""
    return f"Please review the following code and suggest improvements:\n\n{code}"

print("✓ Registered prompt: review_code")

✓ Registered prompt: review_code


### Prompts — *user-controlled*
A prompt is a **reusable template** the **user** explicitly invokes — often surfaced as a slash-command or menu item in the host. It returns one or more messages with the arguments filled in. Prompts package an expert-crafted workflow so the user doesn't have to retype it each time.

## Part 4 — Connect a Client

A client talks to the server over a transport. For a fully in-notebook demo we use the SDK's **in-memory** transport, `create_connected_server_and_client_session`, which wires a client directly to our server object — the exact same `ClientSession` API you'd get over stdio or HTTP, with no subprocess.

The client also supplies **callbacks** for the client-side primitives (sampling, roots, elicitation, logging). A real host backs these with its UI and model; we **mock** them so everything runs offline. We explain each in Part 5.

In [6]:
# Mocked CLIENT capabilities (a real host would back these with its model + UI).

async def sampling_callback(context, params):
    """SAMPLING: the server asked our 'model' to generate. Return a canned reply."""
    last = params.messages[-1].content.text
    return types.CreateMessageResult(
        role="assistant",
        content=types.TextContent(type="text", text=f"[mock completion for: {last[:50]}]"),
        model="mock-model-1",
        stopReason="endTurn",
    )

async def roots_callback(context):
    """ROOTS: advertise which directories the server may operate within."""
    return types.ListRootsResult(roots=[
        types.Root(uri="file:///home/student/project", name="project"),
    ])

async def logging_callback(params):
    """LOGGING: receive log records the server emits."""
    print(f"   [server log · {params.level}] {params.data}")

async def elicitation_callback(context, params):
    """ELICITATION: the server asked the user a question. Auto-answer it."""
    return types.ElicitResult(action="accept", content={"favorite_number": 24})

def client():
    """Open a fresh in-memory client session wired to `server` with our callbacks."""
    return connect_client(
        server,
        sampling_callback=sampling_callback,
        list_roots_callback=roots_callback,
        logging_callback=logging_callback,
        elicitation_callback=elicitation_callback,
    )

print("✓ Client callbacks defined")

✓ Client callbacks defined


In [ ]:
# THE HANDSHAKE: initialize() negotiates capabilities between client and server.
async def demo_initialize():
    async with client() as session:
        init = await session.initialize()
        print("Connected to server:", init.serverInfo.name)
        caps = init.capabilities
        print("Server advertises capabilities:")
        print("  tools:    ", caps.tools)
        print("  resources:", caps.resources)
        print("  prompts:  ", caps.prompts)

await demo_initialize()

In [ ]:
# TOOLS in action: list what the model would see, then call one.
async def demo_tools():
    async with client() as session:
        await session.initialize()

        tools = await session.list_tools()
        print("Available tools:")
        for t in tools.tools:
            args = ", ".join(t.inputSchema["properties"])
            print(f"  • {t.name}({args}) — {t.description}")

        r1 = await session.call_tool("add", {"a": 6, "b": 4})
        print("\ncall_tool add(6, 4) ->", r1.content[0].text)

        r2 = await session.call_tool("verify_arithmetic", {"expression": "6 * 4 = 24"})
        print("call_tool verify_arithmetic('6 * 4 = 24') ->", r2.content[0].text)

await demo_tools()

In [ ]:
# RESOURCES in action: list static resources + templates, then read them.
async def demo_resources():
    async with client() as session:
        await session.initialize()

        res = await session.list_resources()
        print("Static resources:  ", [str(r.uri) for r in res.resources])

        tmpl = await session.list_resource_templates()
        print("Resource templates:", [t.uriTemplate for t in tmpl.resourceTemplates])

        got = await session.read_resource("config://server")
        print("\nread config://server ->", got.contents[0].text.replace("\n", " | "))

        got = await session.read_resource("note://game-of-24")
        print("read note://game-of-24 ->", got.contents[0].text)

await demo_resources()

In [ ]:
# PROMPTS in action: list available prompts, then fetch one with arguments filled in.
async def demo_prompts():
    async with client() as session:
        await session.initialize()

        prompts = await session.list_prompts()
        print("Available prompts:", [p.name for p in prompts.prompts])

        got = await session.get_prompt("review_code", {"code": "def f(x): return x+1"})
        print("\nget_prompt review_code ->")
        for msg in got.messages:
            print(f"   [{msg.role}] {msg.content.text}")

await demo_prompts()

## Part 5 — Client Primitives

So far the *server* offered capabilities to the *client*. MCP is symmetric: the **client** offers three capabilities back to the **server**. These are the parts people most often miss.

- **Sampling** — a server can ask the **host's model** to generate text (`ctx.session.create_message`). This lets a server be "agentic" without shipping its own model or API key, and the host keeps the human in the loop.
- **Roots** — the client advertises **filesystem/URI boundaries** (`ctx.session.list_roots`) so the server knows where it may operate.
- **Elicitation** — a server can **ask the user for input** mid-task (`ctx.elicit`), e.g. a missing parameter or a confirmation.

Each is exercised below by a tool that *uses* the client capability; our mocked callbacks from Part 4 stand in for a real host. Tools reach these via a `Context` argument the SDK injects.

In [ ]:
# SAMPLING — a tool that asks the client's model to summarize text.
@server.tool()
async def summarize(text: str, ctx: Context) -> str:
    """Ask the host's LLM (via sampling) to summarize the given text."""
    result = await ctx.session.create_message(
        messages=[types.SamplingMessage(
            role="user",
            content=types.TextContent(type="text", text=f"Summarize in one line: {text}"),
        )],
        max_tokens=64,
    )
    return result.content.text

async def demo_sampling():
    async with client() as session:
        await session.initialize()
        r = await session.call_tool(
            "summarize", {"text": "MCP standardizes how apps feed context to LLMs."})
        print("summarize(...) ->", r.content[0].text)

await demo_sampling()

In [ ]:
# ROOTS — a tool that reads the client's advertised working directories.
@server.tool()
async def list_workspace(ctx: Context) -> str:
    """Return the roots (boundaries) the client has granted to the server."""
    roots = await ctx.session.list_roots()
    return ", ".join(str(r.uri) for r in roots.roots)

async def demo_roots():
    async with client() as session:
        await session.initialize()
        r = await session.call_tool("list_workspace", {})
        print("list_workspace() ->", r.content[0].text)

await demo_roots()

In [ ]:
# ELICITATION — a tool that asks the user a question mid-execution.
class NumberChoice(BaseModel):
    favorite_number: int

@server.tool()
async def pick_number(ctx: Context) -> str:
    """Ask the user (via elicitation) for a number, then use it."""
    result = await ctx.elicit(message="What's your favorite number?", schema=NumberChoice)
    if result.action == "accept":
        return f"The user chose {result.data.favorite_number}."
    return f"The user did not answer (action={result.action})."

async def demo_elicitation():
    async with client() as session:
        await session.initialize()
        r = await session.call_tool("pick_number", {})
        print("pick_number() ->", r.content[0].text)

await demo_elicitation()

## Logging & Notifications

Beyond request/response, MCP has **notifications** — one-way messages with no reply:

- **Logging** — a server sends log records to the client (`ctx.info`, `ctx.warning`, …). Our `logging_callback` from Part 4 prints them.
- **Progress** — long-running operations report progress (`ctx.report_progress`) so the host can render a progress bar.
- **List-changed** — a server notifies the client when its tools/resources/prompts change (`notifications/tools/list_changed`), prompting a re-fetch.
- **Cancellation** — either side can cancel an in-flight request.

In [7]:
# LOGGING — a tool that emits log records (delivered to logging_callback).
@server.tool()
async def noisy_add(a: int, b: int, ctx: Context) -> int:
    """Add two numbers, logging along the way."""
    await ctx.info(f"received a={a}, b={b}")
    total = a + b
    await ctx.warning("this is just a demo warning")
    return totaly

async def demo_logging():
    async with client() as session:
        await session.initialize()
        r = await session.call_tool("noisy_add", {"a": 2, "b": 3})
        print("noisy_add(2, 3) ->", r.content[0].text)

await demo_logging()

   [server log · info] received a=2, b=3
   [server log · warning] this is just a demo warning
noisy_add(2, 3) -> 5


## Part 6 — On the Wire (JSON-RPC 2.0)

Every call above is JSON-RPC under the hood. For example, `call_tool("add", {"a": 6, "b": 4})` sends:

```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/call",
  "params": { "name": "add", "arguments": { "a": 6, "b": 4 } }
}
```

and the server replies:

```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "content": [ { "type": "text", "text": "10" } ],
    "isError": false
  }
}
```

The other methods follow the same shape: listing tools is `"tools/list"`, reading a resource is `"resources/read"` (with a `uri` param), fetching a prompt is `"prompts/get"`. The SDK builds and parses these for you — but it's all inspectable, plain JSON.

## Part 7 — Running a Real Server

In production the server is its own process. The same `server` object runs over **stdio** with a single line:

```python
# server.py
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("demo-server")

@mcp.tool()
def add(a: int, b: int) -> int:
    return a + b

if __name__ == "__main__":
    mcp.run()            # defaults to the stdio transport
```

Then:
- **Inspect it:** `mcp dev server.py` launches the MCP Inspector to click through tools/resources/prompts.
- **Use it in Claude Desktop:** add it to `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "demo": { "command": "python", "args": ["/absolute/path/to/server.py"] }
  }
}
```

For remote hosting, call `mcp.run(transport="streamable-http")` instead and point the host at the URL. The next cell writes a runnable `demo_server.py` to disk so you can try `mcp dev demo_server.py` in a terminal.

In [ ]:
# Write a standalone server you can run OUTSIDE the notebook (its own process).
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo-server")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@mcp.resource("config://server")
def server_config() -> str:
    return "name=demo-server version=1.0"

@mcp.prompt()
def review_code(code: str) -> str:
    return "Please review this code: " + code

if __name__ == "__main__":
    mcp.run()
'''

with open("demo_server.py", "w") as f:
    f.write(server_code.lstrip())

print("✓ Wrote demo_server.py")
print("  Try it in a terminal:  mcp dev demo_server.py")

## Part 8 — A Simple Authentication Implementation

Whether a server needs authentication depends on its **transport**:

- **stdio** — the server is a local subprocess that *you* launched, so trust is established by the operating system; there is no in-protocol authentication. (The server may still hold its own credentials for downstream APIs it calls.)
- **Streamable HTTP** — the server is reachable over the network, so it **must** authenticate callers. The MCP spec uses **OAuth 2.1**: the MCP server acts as an **OAuth Resource Server** that validates a **bearer token** on every request.

The full OAuth 2.1 flow (authorization server, PKCE, dynamic client registration, protected-resource metadata) is a lot of moving parts. The one piece you always implement is a **`TokenVerifier`**: given the bearer token from the `Authorization` header, return an `AccessToken` (carrying the caller's client id and granted **scopes**) if the token is valid, or `None` to reject the request (→ HTTP 401).

Below is the simplest possible verifier — a static token store. Real deployments validate signed JWTs or introspect the token against an authorization server.

In [ ]:
from mcp.server.auth.provider import TokenVerifier, AccessToken

# A tiny token store: token string -> who they are + what they may do.
KNOWN_TOKENS = {
    "secret-token-abc":   AccessToken(token="secret-token-abc",   client_id="alice", scopes=["mcp:read", "mcp:write"]),
    "readonly-token-xyz": AccessToken(token="readonly-token-xyz", client_id="bob",   scopes=["mcp:read"]),
}

class StaticTokenVerifier(TokenVerifier):
    """Verify a bearer token against an in-memory store (the simplest possible scheme)."""
    async def verify_token(self, token: str) -> AccessToken | None:
        return KNOWN_TOKENS.get(token)   # None  ->  the request is rejected (HTTP 401)

# Verify tokens directly — this is exactly what the server runs on each request.
async def demo_auth():
    verifier = StaticTokenVerifier()
    ok  = await verifier.verify_token("secret-token-abc")
    bad = await verifier.verify_token("not-a-real-token")
    print("valid token   ->", f"client={ok.client_id}, scopes={ok.scopes}")
    print("invalid token ->", bad, "(server would reject with HTTP 401)")

await demo_auth()

In [ ]:
from mcp.server.auth.settings import AuthSettings

# Attach the verifier to an HTTP-facing server. It is now an OAuth "Resource
# Server": every request must carry a valid bearer token.
secure_server = FastMCP(
    "secure-demo-server",
    token_verifier=StaticTokenVerifier(),
    auth=AuthSettings(
        issuer_url="https://auth.example.com",          # the authorization server
        resource_server_url="https://api.example.com",  # this server's public URL
        required_scopes=["mcp:read"],                    # minimum scope to connect
    ),
)

@secure_server.tool()
def whoami() -> str:
    """Return the authenticated caller, read from the request's auth context."""
    from mcp.server.auth.middleware.auth_context import get_access_token
    token = get_access_token()
    if token is None:
        return "not authenticated"
    return f"client={token.client_id}, scopes={token.scopes}"

print("✓ Secured server configured:", secure_server.name)
print("  Serve it with:  secure_server.run(transport='streamable-http')")
print("  Clients then send:  Authorization: Bearer <token>")

### Authorization with scopes

Authentication answers *who are you?*; **authorization** answers *what may you do?* The `AccessToken.scopes` you returned are how a tool enforces that — inside a request, `get_access_token()` exposes the caller's scopes:

```python
from mcp.server.auth.middleware.auth_context import get_access_token

@secure_server.tool()
def delete_record(record_id: int) -> str:
    token = get_access_token()
    if token is None or "mcp:write" not in token.scopes:
        raise PermissionError("requires the 'mcp:write' scope")
    ...  # perform the delete
```

(Over the in-memory transport used in this notebook there is no HTTP request, so `get_access_token()` returns `None`; enforcement happens once the server runs over Streamable HTTP.)

### Where this fits in the full picture
A production deployment wraps the rest of OAuth 2.1 around this verifier: a separate **authorization server** issues tokens, clients obtain them using **PKCE**, and the MCP server publishes **protected-resource metadata** (RFC 9728) telling clients where to authenticate.

### Security checklist
- Serve over **HTTPS only** — bearer tokens are credentials.
- Prefer **short-lived tokens** (set `expires_at`) and **least-privilege scopes**.
- **Never log** raw tokens.
- Reject by default: only accept a token you can positively validate.

## Reflection
1. **Who controls what?** For tools, resources, and prompts, name the controller (model / application / user) and explain why that distinction matters for safety.
2. **Sampling** lets a server call the host's model. Why is that more powerful — and more in need of human oversight — than an ordinary tool?
3. **Roots vs. a path argument:** why expose working directories as *roots* rather than just passing a path argument into every tool?
4. **Elicitation vs. arguments:** when would a server elicit input mid-task instead of declaring an argument up front?
5. **Transports:** when would you choose **stdio** vs. **Streamable HTTP** for a server?
6. **M×N → M+N:** in your own words, what does standardizing on MCP buy an ecosystem of apps and tools?
7. **Authentication:** why does a stdio server typically need no in-protocol auth while a Streamable-HTTP server does? What does a `TokenVerifier` return, and what happens when it returns `None`?

## Summary

### The primitives
| Side | Primitive | Controlled by |
|---|---|---|
| **Server** | Tools | the model |
| | Resources | the application |
| | Prompts | the user |
| **Client** | Sampling | server → host model |
| | Roots | client → server |
| | Elicitation | server → user |

### Mechanics
JSON-RPC 2.0 messages · the `initialize` capability negotiation · stdio & Streamable-HTTP transports · logging, progress, list-changed & cancellation notifications, plus bearer-token authentication (OAuth 2.1 Resource Server).

### Build recipe (FastMCP)
```python
server = FastMCP("name")

@server.tool()                  # model-controlled action
@server.resource("uri://...")   # application-controlled context
@server.prompt()                # user-controlled template

server.run()                    # serve over stdio
```

### Resources
- Spec & docs: https://modelcontextprotocol.io
- Python SDK: https://github.com/modelcontextprotocol/python-sdk
- Reference servers: https://github.com/modelcontextprotocol/servers